Confirmar o ambiente e ler a camada Silver

In [0]:
#Confirmar que o Spart está ativo
print(f"Spark version: {spark.version}")

In [0]:
#Ler direto da tabela Silver registrada no catálogo
df_silver = spark.table("workspace.lakehouse_edu.silver_student_performance")

In [0]:
#Confirmar o que temos
print(f"Total de registros na Silvar: {df_silver.count()}")
print(f"Total de colunas: {len(df_silver.columns)}")

Primeira tabela Gold: Desempenho por escola

In [0]:
from pyspark.sql import functions as F

#Tabela Gold 1 - Desempenho por escola e sexo
#Pergunta de negócio: Qual o desempenho médio por escola, separado por sexto e status de aprovação?
df_gold_escola = (
      df_silver
      .groupBy("escola", "sexo", "status_aprovacao")
      .agg(
          F.count("*").alias("total_alunos"),
          F.round(F.avg("nota_final"), 2).alias("media_nota_final"),
          F.round(F.avg("media_notas"), 2).alias("media_geral"),
          F.round(F.avg("faltas"), 1).alias("media_faltas"),
          F.round(F.avg("horas_estudo_semana"), 1).alias("media_horas_estudo_semana")
      )
      .orderBy("escola","sexo","status_aprovacao")
)

display(df_gold_escola)

In [0]:
#Total por escola para calcular proporção

total_por_escola = (
    df_silver
    .groupBy("escola")
    .agg(F.count("*").alias("total_escola"))
)

In [0]:
#Adicionar percentual proporcional
df_gold_escola = (
    df_gold_escola
    .join(total_por_escola, on="escola", how="left")
    .withColumn("percentual_grupo",
        F.round(F.col("total_alunos") * 100 / F.col("total_escola"), 1)
    )
    .orderBy("escola", "sexo", "status_aprovacao")
)

In [0]:
display(df_gold_escola)

Segunda tabela gold: Fatores de risco de reprovação

In [0]:
df_gold_risco = (
    df_silver
    .groupBy(
        "status_aprovacao",
        "quer_ensino_superior",
        "tem_internet",
        "suporte_escolar"
    )
    .agg(
       F.count("*").alias("total_alunos"),
       F.round(F.avg("nota_final"), 2).alias("media_nota_final"),
       F.round(F.avg("reprovacoes_anteriores"), 2).alias("media_reprovacoes"),
       F.round(F.avg("faltas"), 1).alias("media_faltas"),
       F.round(F.avg("consumo_alcool_semana"), 2).alias("media_alcool_semana")  
    )
    .orderBy("status_aprovacao", F.desc("total_alunos"))
)

display(df_gold_risco)

Terceira tabela gold: Perfil completo por aluno

In [0]:
from pyspark.sql.window import Window

# Definir janela de ranking ordenada pela média de notas
window_ranking = Window.orderBy(F.desc("media_notas"))

df_gold_alunos = (
    df_silver
    .select(
        "escola",
        "sexo",
        "idade",
        "tipo_endereco",
        "horas_estudo_semana",
        "reprovacoes_anteriores",
        "tem_internet",
        "quer_ensino_superior",
        "faltas",
        "nota_1_bimestre",
        "nota_2_bimestre",
        "nota_final",
        "media_notas",
        "status_aprovacao"
    )
    .withColumn("ranking", F.rank().over(window_ranking))
    .withColumn("faixa_desempenho",
        F.when(F.col("media_notas") >= 16, "excelente")
        .when(F.col("media_notas") >= 12, "bom")
        .when(F.col("media_notas") >= 10, "regular")
        .otherwise("insuficiente")
    )
    .orderBy("ranking")
)

display(df_gold_alunos.limit(15))

Salvar as três tabelas Gold no catálogo

In [0]:
# Salvar Gold 1 — Desempenho por escola
(
    df_gold_escola
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.lakehouse_edu.gold_desempenho_por_escola")
)
print("✅ Gold 1 — gold_desempenho_por_escola salva!")

In [0]:
# Salvar Gold 2 — Fatores de risco
(
    df_gold_risco
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.lakehouse_edu.gold_fatores_risco_reprovacao")
)
print("✅ Gold 2 — gold_fatores_risco_reprovacao salva!")

In [0]:
# Salvar Gold 3 — Perfil dos alunos
(
    df_gold_alunos
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.lakehouse_edu.gold_perfil_alunos")
)
print("✅ Gold 3 — gold_perfil_alunos salva!")

In [0]:
# Confirmar todas as tabelas do catálogo
spark.sql("SHOW TABLES IN workspace.lakehouse_edu").display()